# Qwen3.8-27B (W4A16 GPTQ) as a Jev-compatible endpoint on 1x CMP 170HX (SM80) — 1card, vLLM

| Metric | Value |
|---|---|
| Read latency, c=1, warm prefix cache (p50) | 141.7 ms |
| Read latency, c=1, cache-busted (p50) | 133.0 ms |
| Labelled accuracy (42 author-labelled reads, T=1) | 0.571 (95% CI 0.42-0.71) |
| Expected calibration error at T=1 → fitted T | 0.274 → 0.223 |
| Option-order sensitivity (max probability shift) | 0.639, argmax unstable |
| Read determinism (sequential / interleaved / concurrent) | identical / identical / identical |

![reliability and option-order rotation](../assets/charts/2026-09-20-qwen3.8-27b-w4a16-jev-1card-vllm.png)

```
hf download Qwen/Qwen3.8-27B-GPTQ-4bit --local-dir <weights>
```

Model guide: [Qwen3.8-27B](../docs/models/qwen3.8-27b.md) · serving recipe credit: [Kis's DFlash2 notebook](2026-08-30-qwen3.8-27b-w4a16-dflash2-1card-vllm.ipynb)


This notebook shows that **Qwen3.8-27B can serve as a Jev-compatible
endpoint** on one CMP 170HX: a `POST /v1/systemone` request returns a calibrated
probability per option, for `choice`, `noul` (yes/no) and `score` questions,
without generating a single token.

The serving stack is the one Kis established in
[`2026-08-30-qwen3.8-27b-w4a16-dflash2-1card-vllm`](2026-08-30-qwen3.8-27b-w4a16-dflash2-1card-vllm.ipynb)
— his notebook is the reference for *how this checkpoint is served on this
card* (W4A16 on one 170HX, the syv-ai recipe lineage, 180 W-safe, ~140 tok/s
generation). This notebook does **not** re-measure generation; it adds the
classification path on top of that runtime and reports what that path measures.

The Jev contract itself is documented in the
[`jev` branch of kishida's llama.cpp fork](https://github.com/kishida/llama.cpp/blob/jev/docs/jev.md)
and is compatible with TypeSafe System One. Credit for the design — one prompt
evaluation, label logits, `confidence = 1 - normalized entropy`, expected-level
`score`, temperature as a calibration divisor, permutations to average out
option order — belongs to that work. What is ours here is the vLLM
implementation of it and the three findings in section 2 that a vLLM host has
to know.

Executed on a single CMP 170HX (SM80) on 2026-09-20. `LIVE = False`: every
number below is read from the committed receipts under
`results/2026-09-20-qwen3.8-27b-w4a16-jev-1card-vllm/`.


In [1]:
# --- Status cell -------------------------------------------------------
# LIVE = False replays the committed receipts under results/<experiment>/.
# LIVE = True runs the same harness against a running pair of endpoints whose
# URL comes from the environment. Never commit a notebook executed with LIVE = True.
import os

EXPERIMENT = "2026-09-20-qwen3.8-27b-w4a16-jev-1card-vllm"
RESULTS_DIR = os.path.join("..", "results", EXPERIMENT)
RECEIPTS = os.path.join(RESULTS_DIR, "receipts")
LIVE = False
JEV_ENDPOINT_ENV_VAR = "JEV_API"      # the interposer, /v1/systemone
VLLM_ENDPOINT_ENV_VAR = "JEV_UPSTREAM"  # the vLLM OpenAI server

print(f"experiment : {EXPERIMENT}")
print(f"receipts   : {RECEIPTS}")
print(f"LIVE       : {LIVE}")
if LIVE:
    for var in (JEV_ENDPOINT_ENV_VAR, VLLM_ENDPOINT_ENV_VAR):
        if not os.environ.get(var):
            raise RuntimeError(f"LIVE=True but {var} is not set")
    print("endpoints  : from the environment (not printed)")


experiment : 2026-09-20-qwen3.8-27b-w4a16-jev-1card-vllm
receipts   : ../results/2026-09-20-qwen3.8-27b-w4a16-jev-1card-vllm/receipts
LIVE       : False


In [2]:
# --- Helpers ------------------------------------------------------------
import json
import os


def receipt(name):
    # One JSON receipt from this experiment's receipts directory.
    with open(os.path.join(RECEIPTS, name)) as f:
        return json.load(f)


def jsonl(name):
    rows = []
    with open(os.path.join(RECEIPTS, name)) as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows


def text(name):
    with open(os.path.join(RECEIPTS, name), errors="replace") as f:
        return f.read()


from IPython.display import display, Markdown


def render_table(headers, rows):
    lines = ["| " + " | ".join(headers) + " |",
             "|" + "|".join(["---"] * len(headers)) + "|"]
    for row in rows:
        lines.append("| " + " | ".join(str(c) for c in row) + " |")
    display(Markdown("\n".join(lines)))


## 1. TL;DR

**Verdict: the classification contract works on this card, and the label
probabilities are exact and repeatable. The honest reading of them is the open
part:** on 42 author-labelled reads the endpoint is right 57% of the time,
over-confident at T=1 (ECE 0.274), and the answer changes with the order the
options are written in.

Three things a vLLM host has to know, each with a receipt in section 2:

1. **`logprobs_mode` decides whether the label distribution exists at all.**
   vLLM's default (`raw_logprobs`) computes logprobs *before*
   `allowed_token_ids`, so a label-masked read returns the unmasked top-k and
   the labels are simply missing. `--logprobs-mode processed_logprobs` fixes
   it. This is not a documentation footnote — it is the difference between a
   working endpoint and a 502.
2. **The checkpoint's own `generation_config` silently filters the labels.**
   It ships `top_k=20, top_p=0.95`, vLLM adopts those as server defaults, and
   in `processed_logprobs` mode the returned logprobs are post-filter: on a
   four-option question one label is dropped and the survivors are inflated.
   The read path sets `top_p=1.0, top_k=-1` per request.
3. **Option order moves the answer.** Rotating a three-option question's order
   shifts the probability mass by up to 0.639 and flips the argmax. Jev's
   `permutations` option exists for exactly this and the endpoint implements
   it; the un-averaged read is the one to be careful with.

Then the calibration picture, stated plainly: the probabilities are
*measurable* and *reproducible* — identical to the last bit on repeat and
across concurrency — but on a 42-example descriptive set they are
over-confident, and **the fitted temperature does not generalise at this
sample size**. In-sample, fitting T by NLL improves ECE from 0.274 to 0.223;
under leave-one-out fitting it comes out *worse* than leaving T=1 alone (ECE
0.301). Read that as: the fitting loop works and is worth running, but n=42
cannot certify a temperature, and the in-sample number is not the one to
quote.

One caveat on the word *reproducible*: the endpoint serves one read per
request and those are bit-identical on repeat. Reads placed in the **same
batch** are not — two copies of the same prompt agreed exactly over HTTP and
differed by up to 0.067 in logprob when batched together in-process (section
4). Bit-identity is a property of a fixed batch composition, not of the model.


### Pins

From `receipts/env.json`, which records the runtime versions, the model config
hash, the resolved engine configuration and the line of engine log that names
the architecture.


In [3]:
import re

env = receipt("env.json")
weights_gib = None
for line in env["log_excerpt"]:
    m = re.search(r"Model loading took ([\d.]+) GiB", line)
    if m:
        weights_gib = m.group(1)
pins = [
    ("Model (served)", "Qwen3.8-27B, W4A16 GPTQ (checkpoint `config.json` hash in the receipt)"),
    ("Architecture resolved by the engine", env["model"]["architectures"][0]),
    ("Quantization", f"{env['model']['quantization_config']['quant_method']} "
                     f"{env['model']['quantization_config']['bits']}-bit, "
                     f"group_size {env['model']['quantization_config']['group_size']}, "
                     f"desc_act {env['model']['quantization_config']['desc_act']}"),
    ("Linear kernel", "MarlinLinearKernel (from the serve log)"),
    ("Attention backend", "FLASH_ATTN (FlashAttention version 2)"),
    ("vLLM", env["measured"]["vllm"]),
    ("torch / CUDA", f"{env['measured']['torch']} / {env['measured']['cuda']}"),
    ("transformers", env["measured"]["transformers"]),
    ("Hardware", "1x NVIDIA CMP 170HX, 64 GiB HBM2e, SM80"),
    ("Topology", "1card, tensor_parallel_size=1, no speculative decoding"),
    ("Serving shape", f"max_model_len {env['serve']['max_model_len']}, "
                      f"gpu_memory_utilization {env['serve']['gpu_memory_utilization']}, "
                      f"max_logprobs {env['serve']['max_logprobs']}, "
                      f"prefix caching on"),
    ("Jev read mode", "`--logprobs-mode processed_logprobs` (not the default)"),
    ("Weights held by the engine", f"{weights_gib} GiB (from the serve log)"),
]
render_table(["Pin", "Value"], pins)


| Pin | Value |
|---|---|
| Model (served) | Qwen3.8-27B, W4A16 GPTQ (checkpoint `config.json` hash in the receipt) |
| Architecture resolved by the engine | Qwen3_5ForConditionalGeneration |
| Quantization | gptq 4-bit, group_size 32, desc_act False |
| Linear kernel | MarlinLinearKernel (from the serve log) |
| Attention backend | FLASH_ATTN (FlashAttention version 2) |
| vLLM | 0.28.0 |
| torch / CUDA | 2.13.0+cu130 / 13.0 |
| transformers | 5.15.0 |
| Hardware | 1x NVIDIA CMP 170HX, 64 GiB HBM2e, SM80 |
| Topology | 1card, tensor_parallel_size=1, no speculative decoding |
| Serving shape | max_model_len 8192, gpu_memory_utilization 0.9, max_logprobs 128, prefix caching on |
| Jev read mode | `--logprobs-mode processed_logprobs` (not the default) |
| Weights held by the engine | 18.79 GiB (from the serve log) |

### Protocol

*Every* read is one `POST /v1/completions` with `max_tokens=1`: the prompt is
evaluated once and the logprobs of the first generated position are read. The
sampled token is discarded and nothing is generated, so the answer does not
depend on sampling.

| Setting | Value | Why |
|---|---|---|
| `temperature` (upstream) | 1.0 | identity; a requested temperature is applied afterwards as `softmax(logprobs / T)` |
| `top_p` / `top_k` (upstream) | 1.0 / -1 | neutralises the checkpoint's `generation_config` nucleus default |
| `allowed_token_ids` | the option symbols | makes the returned logprobs the label-set distribution |
| `logprobs` | number of labels | one logprob per label; 62 single-token symbols max |
| `return_tokens_as_token_ids` | true | label logprobs are matched by token id, not by text |
| prompt | Jev's format, model chat template | assistant generation prompt, `enable_thinking=False` |

Prompt format, label selection and the temperature/permutations semantics are
the ones in kishida's `jev` docs, with one deviation to be explicit about:
`enable_thinking=False` on this checkpoint still renders an empty
`<think>\n\n</think>\n\n` block before the label position (visible in
`receipts/prompts.json`). The read works because the empty block is part of the
generation prompt and the label follows it, but a prompt built by hand without
it is not the same prompt.


In [4]:
prompts = receipt("prompts.json")
rows = []
for f in prompts["fixtures"]:
    rows.append([f["case"], f["question"]["type"], f["prompt_tokens"],
                 len(f["option_names"]), " ".join(f["label_symbols"]),
                 f["prompt_sha256"][:16] + "..."])
render_table(["Fixture", "Question type", "Prompt tokens", "Options",
              "Label symbols", "Prompt sha256"], rows)
print("rendered prompt for the noul fixture:")
print(prompts["fixtures"][1]["rendered_prompt"])


| Fixture | Question type | Prompt tokens | Options | Label symbols | Prompt sha256 |
|---|---|---|---|---|---|
| choice-billing | choice | 92 | 3 | A B C | 7f190d70226659cc... |
| noul-angry | noul | 69 | 2 | A B | cfca2a21fd905232... |
| score-urgent | score | 78 | 4 | A B C D | 5a2a549445ffaf04... |

rendered prompt for the noul fixture:
<|im_start|>user
Context:
Third time I am writing. My order still has not arrived. Refund me.

Answer the question with only the label of the best option (the character before the colon), nothing else.
Question: Is this customer angry?
Options:
yes
no<|im_end|>
<|im_start|>assistant
<think>

</think>




## 2. Visible results

Every table and the chart are computed from the committed receipts. Raw
payloads, including full logprob dictionaries, are in the receipts themselves.


### 2.1 The finding that decides the whole thing: `logprobs_mode`

The same prompt, read twice with `allowed_token_ids=[labels]`. With vLLM's
default mode the payload holds the **unmasked** ranking — the labels are not in
it, so the endpoint cannot answer at all. With `processed_logprobs` the payload
is exactly the label set, and the mass over it sums to 1.

`labels_visible_unmasked` counts how many labels also appear in an unmasked
top-20 read; where they do, the *pairwise differences* of the logprobs are
identical between the two reads (they differ only by the normalisation
constant), which is the invariant that proves the whitelist precedes the
gather rather than following it.


In [5]:
raw = receipt("mask-raw_logprobs.json")
proc = receipt("mask-processed_logprobs.json")

rows = []
for r in raw["rows"]:
    rows.append(["raw_logprobs (default)", r["case"], len(r["masked_returned_ids"]),
                 r["returned_ids_equal_allowed"], r["count_equals_n_labels"],
                 "not auditable"])
for r in proc["rows"]:
    rows.append(["processed_logprobs", r["case"], len(r["masked_returned_ids"]),
                 r["returned_ids_equal_allowed"], r["count_equals_n_labels"],
                 f"{r['sum_exp_masked']:.6f}"])
render_table(["Mode", "Fixture", "Logprobs returned", "Returned == allowed",
              "Count == labels", "Mass over labels"], rows)

shared = [r for r in proc["rows"] if r["labels_visible_unmasked"]]
print("offset-invariance where labels are visible in both reads:")
for r in shared:
    print(f"  {r['case']}: {r['labels_visible_unmasked']} shared labels, "
          f"max |Δ(lp_i - lp_j)| = {r['max_abs_offset_delta']:.3e}")


| Mode | Fixture | Logprobs returned | Returned == allowed | Count == labels | Mass over labels |
|---|---|---|---|---|---|
| raw_logprobs (default) | choice-billing | 4 | False | False | not auditable |
| raw_logprobs (default) | noul-angry | 3 | False | False | not auditable |
| raw_logprobs (default) | score-urgent | 5 | False | False | not auditable |
| processed_logprobs | choice-billing | 3 | True | True | 1.000000 |
| processed_logprobs | noul-angry | 2 | True | True | 1.000000 |
| processed_logprobs | score-urgent | 4 | True | True | 1.000000 |

offset-invariance where labels are visible in both reads:
  noul-angry: 2 shared labels, max |Δ(lp_i - lp_j)| = 2.980e-08
  score-urgent: 2 shared labels, max |Δ(lp_i - lp_j)| = 0.000e+00


### 2.2 The checkpoint's `generation_config` filters the labels

vLLM adopts the model's `generation_config.json` (`temperature 1.0, top_k 20,
top_p 0.95`) as the server's default sampling parameters — the serve log says
so. In `processed_logprobs` mode the returned logprobs are post-filter, so a
read that does not neutralise those defaults is a nucleus read: on the
four-option question the low-mass label is **dropped entirely** and the
surviving probabilities are inflated.

This is silent. Nothing errors; the endpoint returns a probability
distribution that is simply not the one the model computed.


In [6]:
nuc = receipt("nucleus.json")
rows = []
for r in nuc["rows"]:
    g = r["generation_config_default"]
    rows.append([r["case"], r["question_type"],
                 len(g["label_logprobs"]), len(g["dropped_labels"]),
                 ", ".join(map(str, g["dropped_labels"])) or "none",
                 f"{r['max_abs_probability_delta']:.4f}"
                 if r["max_abs_probability_delta"] is not None else "labels dropped"])
render_table(["Fixture", "Question type", "Labels returned under defaults",
              "Labels dropped", "Dropped ids",
              "Max |Δprobability| vs neutralised"], rows)
print("neutralised vs generation_config defaults, per fixture:")
for r in nuc["rows"]:
    print(f"  {r['case']}: neutralised {[round(p, 4) for p in r['neutralised']['probabilities']]}"
          f"  dropped={r['generation_config_default']['dropped_labels']}")


| Fixture | Question type | Labels returned under defaults | Labels dropped | Dropped ids | Max |Δprobability| vs neutralised |
|---|---|---|---|---|---|
| choice-billing | choice | 3 | 0 | none | 0.0000 |
| noul-angry | noul | 2 | 0 | none | 0.0000 |
| score-urgent | score | 4 | 0 | none | 0.0000 |
| choice-longshot | choice | 4 | 1 | 35 | labels dropped |

neutralised vs generation_config defaults, per fixture:
  choice-billing: neutralised [0.5209, 0.2619, 0.2172]  dropped=[]
  noul-angry: neutralised [0.7186, 0.2814]  dropped=[]
  score-urgent: neutralised [0.4208, 0.0778, 0.0535, 0.4479]  dropped=[]
  choice-longshot: neutralised [0.6885, 0.1124, 0.1536, 0.0454]  dropped=[35]


### 2.3 Determinism: the same prompt gives the same distribution

Three reads back to back, two reads around a different prompt, and two reads
fired concurrently. The label logprobs are compared; the sampled token is not
part of the answer, so its randomness is irrelevant.

This is a claim about an idle, single-tenant server at c=1. Continuous
batching changes kernel shapes, and some kernels are not batch-invariant, so
"identical under concurrency" here is measured for two concurrent requests and
is not a licence to assume it at high concurrency. The appendix measures the
effect directly: two copies of the same prompt agree exactly when they are
separate requests and differ by up to 0.067 in logprob when they share a
batch. Bit-identity belongs to a batch composition, not to the model.


In [7]:
det = receipt("determinism.json")
rows = [
    ["Sequential (3 reads, back to back)", det["sequential_identical"],
     "exact equality of label logprobs"],
    ["Interleaved (a different prompt in between)", det["interleaved_identical"],
     "same"],
    ["Concurrent (2 in flight)", det["concurrent_identical"],
     f"max |Δlogprob| = {det['concurrent_max_abs_delta']:.1e}"],
]
render_table(["Condition", "Identical", "Note"], rows)
print("label logprobs, sequential reads:")
for i, lp in enumerate(det["sequential_logprobs"]):
    print(f"  read {i}: yes {lp['yes']:.10f}  no {lp['no']:.10f}")


| Condition | Identical | Note |
|---|---|---|
| Sequential (3 reads, back to back) | True | exact equality of label logprobs |
| Interleaved (a different prompt in between) | True | same |
| Concurrent (2 in flight) | True | max |Δlogprob| = 0.0e+00 |

label logprobs, sequential reads:
  read 0: yes -0.3304581940  no -1.2679581642
  read 1: yes -0.3304581940  no -1.2679581642
  read 2: yes -0.3304581940  no -1.2679581642


### 2.4 Temperature is a calibration divisor, applied after the read

Every read runs at T=1. A requested temperature is applied to the label
logprobs as `softmax(logprobs / T)`, so the same read serves any calibration
and changing T cannot change what was read. Note what T does and does not
move: it cannot change the argmax (dividing by a positive scalar preserves
order), but it does change `confidence` **and** the expected level of a
`score` question, because a flatter distribution shifts the probability-
weighted mean. The table checks the endpoint's probabilities against that
formula computed client-side.


In [8]:
temp = receipt("temperature.json")
rows = []
for case, blob in temp["cases"].items():
    for r in blob["rows"]:
        rows.append([case, blob["question_type"], r["temperature"],
                     " ".join(f"{p:.6f}" for p in r["api_probabilities"]),
                     f"{r['max_abs_delta']:.1e}"])
render_table(["Fixture", "Question type", "T",
              "Probabilities at T", "|API - client formula|"], rows)


| Fixture | Question type | T | Probabilities at T | |API - client formula| |
|---|---|---|---|---|
| choice-billing | choice | 0.5 | 0.700961 0.177231 0.121809 | 0.0e+00 |
| choice-billing | choice | 1.0 | 0.520916 0.261933 0.217150 | 0.0e+00 |
| choice-billing | choice | 2.0 | 0.424673 0.301138 0.274189 | 0.0e+00 |
| choice-billing | choice | 4.0 | 0.377985 0.318295 0.303720 | 0.0e+00 |
| noul-angry | noul | 0.5 | 0.867036 0.132964 | 8.3e-17 |
| noul-angry | noul | 1.0 | 0.718594 0.281406 | 0.0e+00 |
| noul-angry | noul | 2.0 | 0.615088 0.384912 | 5.6e-17 |
| noul-angry | noul | 4.0 | 0.558327 0.441673 | 5.6e-17 |
| score-urgent | score | 0.5 | 0.457974 0.015671 0.007402 0.518953 | 0.0e+00 |
| score-urgent | score | 1.0 | 0.420767 0.077834 0.053495 0.447904 | 0.0e+00 |
| score-urgent | score | 2.0 | 0.354811 0.152602 0.126512 0.366074 | 0.0e+00 |
| score-urgent | score | 4.0 | 0.305933 0.200636 0.182681 0.310751 | 0.0e+00 |

### 2.5 Option order changes the answer

A three-option routing question, read once per rotation of the option order.
The probability read for "billing" stays between 0.481 and 0.536 — but it wins
in one rotation and loses in the other two, because the gap between the top two
options is smaller than the shift the order induces. Jev's `permutations`
option averages over rotations; the endpoint implements it and the last row
confirms the averaged answer is the mean of the individually measured
rotations.


In [9]:
perm = receipt("permutations.json")
rows = []
for r in perm["per_rotation"]:
    order = r["option_order"]
    pretty = "  ".join(f"{n}={p:.4f}" for n, p in zip(order, r["probabilities"]))
    rows.append([r["rotation"], ", ".join(order), pretty,
                 f"{r['l1_vs_rotation0']:.4f}", r["argmax_vs_rotation0"]])
render_table(["Rotation", "Order presented", "Probability per option",
              "L1 vs rotation 0", "Argmax same"], rows)
print(f"argmax stable across rotations: {perm['argmax_stable']}")
print(f"API permutations={len(perm['per_rotation'])} vs mean of measured rotations: "
      f"max |Δ| = {perm['api_vs_mean_max_abs_delta']:.4f}")


| Rotation | Order presented | Probability per option | L1 vs rotation 0 | Argmax same |
|---|---|---|---|---|
| 0 | billing, technical, sales | billing=0.5209  technical=0.2619  sales=0.2172 | 0.0000 | True |
| 1 | technical, sales, billing | technical=0.4810  sales=0.2918  billing=0.2272 | 0.5874 | False |
| 2 | sales, billing, technical | sales=0.5365  billing=0.2534  technical=0.2101 | 0.6387 | False |

argmax stable across rotations: False
API permutations=3 vs mean of measured rotations: max |Δ| = 0.0038


### 2.6 Labelled reads and calibration

A 42-example author-labelled set (`receipts/labeled.jsonl`): 20 routing
questions over billing/technical/sales, 14 sentiment yes/no, 8 urgency levels.
Labels are the author's reading of each text, not a gold corpus; the set is
small and in-domain, so read the numbers as descriptive.

Reported: accuracy with a Wilson 95% interval, multiclass Brier, NLL of the
true class, and 10-bin ECE at T=1 and at the NLL-fitted temperature. The
leave-one-out row fits the temperature on the other 41 examples for each held
out example, which is the honest version of the ECE — and it is **worse than
not fitting at all** (0.301 against 0.274). With 42 examples a single
temperature does not generalise; the in-sample gain (0.223) is the fit
absorbing its own sample. Anything built on this should either fit T on a
separate, larger calibration split or leave it at 1.


In [10]:
m = receipt("metrics.json")
rows = []
for key, label in (("at_T1", "T = 1 (as served)"),
                   ("at_fitted_T", f"T = {m['fitted_temperature']:.2f} (fitted)")):
    d = m[key]
    rows.append([label, f"{d['accuracy']:.3f}",
                 f"{d['accuracy_wilson95'][0]:.3f}-{d['accuracy_wilson95'][1]:.3f}",
                 f"{d['brier_multiclass']:.4f}", f"{d['nll_true_class']:.4f}",
                 f"{d['ece_10bin']:.4f}"])
loo = m["leave_one_out"]
rows.append(["leave-one-out fitting", f"{loo['accuracy']:.3f}", "-", "-", "-",
             f"{loo['ece_10bin']:.4f}"])
render_table(["Reads", "Accuracy", "95% CI", "Brier", "NLL", "ECE (10 bins)"], rows)
print("by question type at T=1:")
for t, d in m["by_question_type_at_T1"].items():
    print(f"  {t:8s} n={d['n']:2d}  accuracy {d['accuracy']:.3f}  "
          f"ECE {d['ece_10bin']:.3f}")
print(f"\nfitted temperature: {m['fitted_temperature']:.3f} "
      "(NLL-minimising on the same 42 examples)")


| Reads | Accuracy | 95% CI | Brier | NLL | ECE (10 bins) |
|---|---|---|---|---|---|
| T = 1 (as served) | 0.571 | 0.422-0.709 | 0.5494 | 0.9543 | 0.2736 |
| T = 2.16 (fitted) | 0.571 | 0.422-0.709 | 0.5356 | 0.8747 | 0.2228 |
| leave-one-out fitting | 0.571 | - | - | - | 0.3010 |

by question type at T=1:
  choice   n=20  accuracy 0.600  ECE 0.329
  noul     n=14  accuracy 0.500  ECE 0.266
  score    n= 8  accuracy 0.625  ECE 0.339

fitted temperature: 2.161 (NLL-minimising on the same 42 examples)


### 2.7 Latency

Thirty reads of one three-option question with a warm prefix cache, then
thirty with the prefix cache reset before each read. The cache-busting uses
`POST /reset_prefix_cache` rather than a prompt nonce, because a nonce changes
the prompt and therefore measures something else.

These are client-side end-to-end wall times at c=1 against a single-tenant
server, and the card is shared with no other job. They do not generalise to
concurrency or to a different card.


In [11]:
lat = receipt("latency.json")
rows = []
for key, label in (("warm", "Warm prefix cache"), ("busted", "Cache reset per read")):
    d = lat[key]
    rows.append([label, d["n"], f"{d['p50_ms']:.1f}", f"{d['p95_ms']:.1f}",
                 f"{d['min_ms']:.1f}", f"{d['max_ms']:.1f}"])
render_table(["Condition", "Reads", "p50 (ms)", "p95 (ms)", "min (ms)", "max (ms)"],
             rows)
print("GPU during the warm run (util %, SM clock, power, temp):", lat["gpu_mid_run"])


| Condition | Reads | p50 (ms) | p95 (ms) | min (ms) | max (ms) |
|---|---|---|---|---|---|
| Warm prefix cache | 30 | 141.7 | 160.3 | 122.8 | 166.9 |
| Cache reset per read | 30 | 133.0 | 150.6 | 116.8 | 188.3 |

GPU during the warm run (util %, SM clock, power, temp): 5 %, 1140 MHz, 40.36 W, 42
39 %, 1470 MHz, 64.10 W, 46


### 2.8 Question types and rejects

`choice`, `noul` and `score` in one request each, and the endpoint's error
behaviour. Every negative case returns the status a client should expect, and
the last row is the engine's own cap, not the interposer's.


In [12]:
neg = receipt("negatives.json")
rows = [[c["case"], c["status"], c["expected_status"],
         "ok" if c["status"] == c["expected_status"] else "MISMATCH"]
        for c in neg["cases"]]
render_table(["Case", "Status", "Expected", ""], rows)
print("all negatives as expected:", neg["all_as_expected"])


| Case | Status | Expected |  |
|---|---|---|---|
| unknown_type | 422 | 422 | ok |
| missing_instructions | 422 | 422 | ok |
| empty_choice_criteria | 422 | 422 | ok |
| score_single_level | 422 | 422 | ok |
| images_unsupported | 422 | 422 | ok |
| bad_temperature | 422 | 422 | ok |
| bad_permutations | 422 | 422 | ok |
| too_many_options | 422 | 422 | ok |
| upstream_logprobs_over_cap | 400 | 400 | ok |

all negatives as expected: True


## 3. Reproduce

**Hardware.** One NVIDIA CMP 170HX (SM80, 64 GiB HBM2e) with forced airflow,
180 W power cap is enough. No second card is needed for serving; the
cross-check in section 4 optionally uses one.

**Weights.** The W4A16 GPTQ checkpoint, 18.79 GiB held by the engine on load:

```bash
hf download Qwen/Qwen3.8-27B-GPTQ-4bit --local-dir <weights>
```

Verify before serving that the config declares `quantization_config.method ==
"gptq"`, 4-bit, `group_size 32`, `desc_act false` — that is the shape the
Marlin kernel serves on SM80.

**Serve.** The runtime is the project's SM80 vLLM build in a venv
(`<venv>/bin/vllm`, version and CUDA in `receipts/env.json`). Two flags are not
defaults and both are required for the Jev path:

```bash
CUDA_VISIBLE_DEVICES=0 VLLM_NO_USAGE_STATS=1 DO_NOT_TRACK=1 \
FLASHINFER_DISABLE_VERSION_CHECK=1 \
<venv>/bin/vllm serve <weights> \
  --served-model-name qwen3.8-27b-jev \
  --port 18030 \
  --gpu-memory-utilization 0.90 \
  --max-model-len 8192 \
  --max-logprobs 128 \
  --logprobs-mode processed_logprobs \
  --enable-prefix-caching
```

`--logprobs-mode processed_logprobs` is the finding in section 2.1. Without it
the endpoint cannot read labels at all. `FLASHINFER_DISABLE_VERSION_CHECK=1`
silences a `flashinfer-cubin`/`flashinfer` version mismatch that aborts engine
startup (`cubin 0.6.13` vs `flashinfer 0.6.16.post3`) — the same workaround
Kis's recipe uses.

**Interposer.** `harness/jev_server.py` is the Jev endpoint; it needs the model
directory only for the tokenizer:

```bash
<venv>/bin/python harness/jev_server.py \
  --upstream http://127.0.0.1:18030 \
  --tokenizer <weights> \
  --model qwen3.8-27b-jev \
  --port 18031
```

**Expected boot time.** Model load 115 s, engine init and CUDA-graph capture
about 6 minutes total on a warm compile cache; the first request after that
pays a Triton JIT for the read shape (1.6 s), then reads settle at ~140 ms.

**Measurements.** Every receipt in this notebook is produced by
`harness/jev_probe.py`, one probe per file:

```bash
MODEL_DIR=<weights> <venv>/bin/python harness/jev_probe.py \
    env prompts mask nucleus determinism temperature permutations latency \
    negatives labeled
```

`make_chart.py` rebuilds the figure from the receipts. A smoke run of the same
stack, without the receipts, is one request:

```bash
curl -s http://127.0.0.1:18031/v1/systemone -H 'Content-Type: application/json' -d '{
  "model": "qwen3.8-27b-jev",
  "state": "I have been trying to connect my Stripe account for 3 days and it keeps failing.",
  "questions": {"department": {"type": "choice",
     "instructions": "Which team should handle this",
     "criteria": {"billing": "Payment or subscription issues",
                  "technical": "Bugs or integration problems",
                  "sales": "Pricing or account questions"}}},
  "options": {"temperature": 1.0, "return_logits": true}}'
```


## 4. Appendix

<details>
<summary>Failed attempts, the engine-vs-API cross-check, safety notes, and limitations (click to expand)</summary>


### Failures on the way here

Every one of these is preserved because each cost time and each has a fix.

| # | Symptom | Cause | Fix |
|---|---|---|---|
| 1 | `402`-style refusal at boot: `Free memory on device cuda:0 (35.43/63.53 GiB) ... less than desired (0.9, 57.17 GiB)` | another job on the card held ~12.7 GiB per GPU | size `--gpu-memory-utilization` to the free memory, or serve on the other card |
| 2 | `RuntimeError: flashinfer-cubin version (0.6.13) does not match flashinfer version (0.6.16.post3)` at engine start | the venv's two flashinfer packages are out of step | `FLASHINFER_DISABLE_VERSION_CHECK=1` (Kis's recipe does the same) |
| 3 | Interposer answered `502 upstream_error: upstream returned no logprobs for label ids [32, 33]` | vLLM's default `raw_logprobs` mode computes logprobs before `allowed_token_ids` | `--logprobs-mode processed_logprobs` |
| 4 | The same 502 with the mode fixed | logprob keys are `token_id:32` in this build, not `token:32`; the parser matched the wrong prefix | parse the integer after the last `:` (both spellings) |
| 5 | Labels missing from an otherwise working read (`choice-longshot`, all four options) | the checkpoint's `generation_config` ships `top_k=20, top_p=0.95` and vLLM adopts it; post-filter logprobs drop the low-mass label | send `top_p=1.0, top_k=-1` on the read |
| 6 | Orphaned `VLLM::EngineCore` holding ~15.7 GiB after a killed boot, making the next boot fail on free memory | engine core survives when the API server is killed mid-startup | kill the engine core before relaunching |
| 7 | `serve.sh` exits and takes the engine with it | the launcher backgrounds its children and returns | run the engine and the interposer as supervised processes, not as background children of a script that exits |


### Engine-vs-API cross-check, and what batching does to a read

The interposer trusts that the HTTP completions path returns what the engine
computes. `crosscheck.json` reads the same prompts twice — in-process with
vLLM's `LLM.generate(...)`, all fixtures in **one batch**, and over HTTP, one
request per read — and compares the label logprobs. `analyze_crosscheck.py`
derives the two questions this answers.

Two of the fixtures carry *the same prompt* as another fixture, which turns the
receipt into a batching probe: identical prompts agree exactly over HTTP (they
are separate requests) and disagree in-process (they sit at different batch
positions). The differences are small — up to ~8e-2 in logprob — and they are
the engine's batch-composition dependence, not a transport defect: nothing
about the HTTP path loses information, and the reads the endpoint serves are
one per request.


In [13]:
ca = receipt("crosscheck-analysis.json")
rows = [[r["case"], r["labels"], f"{r['max_abs_delta_http_vs_inprocess']:.4f}"
         if r["max_abs_delta_http_vs_inprocess"] is not None else "n/a",
         r["inprocess_only"], r["http_only"]] for r in ca["transport_fidelity"]["rows"]]
render_table(["Fixture", "Labels", "|Δ logprob| HTTP vs in-process (batched)",
              "In-process only", "HTTP only"], rows)
print(f"exact matches: {ca['transport_fidelity']['exact_matches']} of "
      f"{ca['transport_fidelity']['of']}")

rows = []
for r in ca["batch_position_sensitivity"]["rows"]:
    rows.append([r["prompt"], r["duplicate_fixture"], r["same_prompt"],
                 f"{r['max_abs_delta_inprocess_pair']:.4f}",
                 f"{r['max_abs_delta_http_pair']:.4f}"])
render_table(["Prompt", "Repeated as", "Same label ids",
              "|Δ| between the two, in one batch",
              "|Δ| between the two, over HTTP"], rows)
print("in-process, same prompt at two batch positions, choice fixture:")
for lp in ca["batch_position_sensitivity"]["rows"][0]["inprocess_logprobs"]:
    print("   ", {k: round(v, 6) for k, v in lp.items()})
print("over HTTP, the same two requests:")
for lp in ca["batch_position_sensitivity"]["rows"][0]["http_logprobs"]:
    print("   ", {k: round(v, 6) for k, v in lp.items()})


| Fixture | Labels | |Δ logprob| HTTP vs in-process (batched) | In-process only | HTTP only |
|---|---|---|---|---|
| choice-billing | 3 | 0.0000 | [] | [] |
| noul-angry | 2 | 0.0000 | [] | [] |
| score-urgent | 4 | 0.0849 | [] | [] |
| choice-billing-rot2 | 3 | 0.0671 | [] | [] |
| noul-angry-2 | 2 | 0.0453 | [] | [] |

exact matches: 2 of 5


| Prompt | Repeated as | Same label ids | |Δ| between the two, in one batch | |Δ| between the two, over HTTP |
|---|---|---|---|---|
| choice-billing | choice-billing-rot2 | True | 0.0671 | 0.0000 |
| noul-angry | noul-angry-2 | True | 0.0453 | 0.0000 |

in-process, same prompt at two batch positions, choice fixture:
    {'32': -0.652166, '33': -1.339666, '34': -1.527166}
    {'32': -0.594225, '33': -1.406725, '34': -1.594225}
over HTTP, the same two requests:
    {'32': -0.652166, '33': -1.339666, '34': -1.527166}
    {'32': -0.652166, '33': -1.339666, '34': -1.527166}


### What this does not show

- **Calibration.** n=42 in-domain author-labelled examples with an in-sample
  temperature fit. The ECE improvement is a demonstration of Jev's fitting
  loop, not certification — and the leave-one-out row (0.301) shows the fitted
  T is worse than T=1 on unseen examples at this sample size. A calibration
  claim needs hundreds of examples and a held-out split; until then, treat the
  fitted temperature as unfitted.
- **Generation.** This notebook serves reads. Throughput, TTFT and
  speculative-decode behaviour for *generation* on this checkpoint and card
  are Kis's notebook's subject, not this one's.
- **Images.** The endpoint returns 422 `images_not_supported`; this checkpoint
  is served text-path only.
- **High concurrency.** Determinism is measured for two concurrent requests on
  an idle server. Batch-composition-dependent kernels can change numerics at
  higher concurrency, and the appendix measures that effect (~0.07 logprob on
  two copies of one prompt sharing a batch). The latency table is c=1 and does
  not generalise to a loaded server.
- **Other models.** The `processed_logprobs` and `generation_config` findings
  are properties of a vLLM host, so they should hold for any checkpoint served
  this way — but they were measured on this one checkpoint and this engine
  build.

### Limitations of the reading itself

- A read costs one forward pass over the prompt (~140 ms here at ~92-100
  prompt tokens). It is not free, and it is not a generation.
- The label set is capped at 62 (single-token symbols `A-Z a-z 0-9`); the
  engine's `--max-logprobs` must be at least the option count, and the
  interposer rejects larger option sets with 422 rather than truncating.
- `score` questions report the expected level under the label distribution;
  they are not a regression and inherit the same over-confidence.

### Safety

Card stayed within the 80 C core / 85 C memory stop conditions during the
serving window; no Xid, ECC or GPU-disappearance events. The card ran at its
180 W cap. Forced airflow was in place for the whole run.

### Evidence

- Receipts, harness, and the raw payloads:
  `results/2026-09-20-qwen3.8-27b-w4a16-jev-1card-vllm/`
- The serving recipe this builds on (credit: Kis):
  [`2026-08-30-qwen3.8-27b-w4a16-dflash2-1card-vllm`](2026-08-30-qwen3.8-27b-w4a16-dflash2-1card-vllm.ipynb)
- The Jev contract:
  [`kishida/llama.cpp`, branch `jev`, `docs/jev.md`](https://github.com/kishida/llama.cpp/blob/jev/docs/jev.md)
</details>
